In [3]:
import os
from dotenv import load_dotenv
from pathlib import Path

def load_environment_variables():
    """
    Loads the environment variables from .env.
    Loads OpenAI api key and target model name used in the experiment
    New environment variables as necessary will be appended
    """

    PROJECT_ROOT = Path.cwd().parent
    ENV_FILE = PROJECT_ROOT / ".env"

    if ENV_FILE.exists():
        print(
            f".env file located at: {ENV_FILE.resolve()}")
        
        load_dotenv()
        env = {
                "api_key":              os.getenv("OPENAI_API_KEY"),
                "model_name":           os.getenv("OPENAI_MODEL_NAME"),
                "input_token_price":    float(os.getenv('MODEL_INPUT_TOKEN_PRICE')),
                "output_token_price":   float(os.getenv('MODEL_OUTPUT_TOKEN_PRICE'))
            }

    else:
        print(
            f".env not found at: {ENV_FILE.resolve()}"
        )
        env = {
            "api_key":              None,
            "model_name":           None,
            "input_token_price":    None,
            "output_token_price":   None
        }

    return env

In [4]:
from openai import OpenAI

env = load_environment_variables()

client = OpenAI(
    api_key=env['api_key']
)

response = client.chat.completions.create(
    model=env['model_name'],
    messages=[
        {
            "role": "user",
            "content": "what is RAG?"
        }
    ]
)

print(response.choices[0].message.content)

.env file located at: D:\Bits-lab\eduassist-llmops-prompt-evaluation\.env
RAG can refer to different concepts depending on the context. Here are a few common interpretations:

1. **RAG Status**: In project management, RAG refers to a system for assessing and communicating the status of a project or task using colored indicators: Red, Amber (or Yellow), and Green. 
   - **Red** indicates that there are significant issues that need immediate attention.
   - **Amber** signifies that there are some concerns that may require monitoring or action.
   - **Green** shows that the project is on track and performing well.

2. **RAG (Retrieve and Generate)**: In the context of machine learning and natural language processing, RAG is a model architecture designed for tasks involving both information retrieval and text generation. It typically retrieves relevant information from a knowledge base before generating responses, making it suitable for answering questions or generating text based on exter

In [5]:
response = client.chat.completions.create(
    model=env['model_name'],
    messages=[
        {
            "role": "user",
            "content": "what is RAG in Natural Language Processing?"
        }
    ]
)

print(response.choices[0].message.content)

RAG in Natural Language Processing stands for "Retrieval-Augmented Generation." It is a framework that combines two key approaches: information retrieval and text generation. The primary motivation behind RAG is to enhance the capabilities of language models by allowing them to access external knowledge sources in real-time. 

Here's a breakdown of how RAG works:

1. **Retrieval Component**: Given an input query or prompt, a retrieval system searches a knowledge base or a set of documents to find relevant information. This corpus can be made up of articles, books, or any text data that contains pertinent information related to the query.

2. **Generation Component**: After retrieving the relevant documents, a text generation model (often a transformer-based model like GPT-3) uses the retrieved information to generate a more informed and contextually relevant response.

3. **Integration**: The integration of these components allows the model to produce outputs that are not only coherent

In [6]:
questions = [
    "what is overfitting in machine learning?",
    "Explain gradient descent in simple terms.",
    "what is neural network?",
    "Difference between supervised and unsupervised learning?",
    "Explain backpropagation in neural networks."
]

In [7]:
PROMT_V1 = """
Answer the following student questions clearly.
Question: {Question}
"""

In [8]:
import time
import pandas as pd

def evaluate_prompt(
        questions,
        prompt_template,
        prompt_version,
        model,
        input_token_price,
        output_token_price
):
    results = []

    for idx, question in enumerate(questions, start=1):

        prompt = prompt_template.format(
            Question=question
        )
        
        start_time = time.time()

        response = client.chat.completions.create(
            model=model,
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        )

        end_time = time.time()

        latency = end_time - start_time

        # token cost calculation
        input_token_price = env['input_token_price']
        output_token_price = env['output_token_price']

        input_cost = (
            response.usage.prompt_tokens / (1024 * 1024) 
        ) * input_token_price

        output_cost = (
            response.usage.completion_tokens / (1024 * 1024)
        ) * output_token_price

        total_cost = input_cost + output_cost
        
        results.append(
            {
                "prompt": prompt,
                "response": response.choices[0].message.content,
                "question_id": idx,
                "prompt_version": prompt_version,
                "model": "gpt-4.1-mini",
                "latency_seconds": latency,
                "input_tokens": response.usage.prompt_tokens,
                "output_tokens": response.usage.completion_tokens,
                "total_tokens": response.usage.total_tokens,
                "input_token_price": input_cost,
                "output_token_price": output_cost,
                "total_cost": total_cost,
                "response_status": "success"


            }
        )

    return pd.DataFrame(results)

In [9]:
df_v1 = evaluate_prompt(
    questions,
    PROMT_V1,
    "v1",
    env['model_name'],
    env['input_token_price'],
    env['output_token_price']
)

print("Statistics of V1 evaluation")
print(df_v1['latency_seconds'].describe())

latency_stats_v1 = df_v1['latency_seconds'].agg(
    ['mean', 'median', "min", 'max', 'std']
).to_frame("value")

print("Selected parameters")
print(latency_stats_v1)

for _, row in df_v1.iterrows():
    print("PROMPT:")
    print(row["prompt"])

    print("\nRESPONSE:")
    print(row["response"])

    print("\n" + "="*80 + "\n")

Statistics of V1 evaluation
count    5.000000
mean     4.537414
std      1.508079
min      3.130252
25%      4.080577
50%      4.083337
75%      4.279506
max      7.113399
Name: latency_seconds, dtype: float64
Selected parameters
           value
mean    4.537414
median  4.083337
min     3.130252
max     7.113399
std     1.508079
PROMPT:

Answer the following student questions clearly.
Question: what is overfitting in machine learning?


RESPONSE:
Overfitting in machine learning refers to a modeling error that occurs when a machine learning model learns not only the underlying patterns in the training data but also the noise and random fluctuations. As a result, the model becomes too complex, capturing irrelevant details, which makes it perform very well on the training dataset but poorly on unseen data (validation or test datasets).

In simple terms, an overfit model essentially memorizes the training data rather than generalizing from it. This lack of generalization can lead to high 

In [10]:
def add_cost_metrics(df, env):
    """
    Function to calculate and add cost metric to input data frame.
    The method assumes that the dataframe contains input and output token price.
    """
    input_token_price = env['input_token_price']
    output_token_price = env['output_token_price']

    df['input_cost'] = (
        df['input_tokens'] / (1024 * 1024) 
    ) * input_token_price

    df['output_cost'] = (
        df['output_tokens'] / (1024 * 1024)
    ) * output_token_price

    df['total_cost'] = df['input_cost'] + df['output_cost']

    return df

In [11]:
#df_v1 = add_cost_metrics(df=df_v1, env=env)

df_v1.info()

<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   prompt              5 non-null      str    
 1   response            5 non-null      str    
 2   question_id         5 non-null      int64  
 3   prompt_version      5 non-null      str    
 4   model               5 non-null      str    
 5   latency_seconds     5 non-null      float64
 6   input_tokens        5 non-null      int64  
 7   output_tokens       5 non-null      int64  
 8   total_tokens        5 non-null      int64  
 9   input_token_price   5 non-null      float64
 10  output_token_price  5 non-null      float64
 11  total_cost          5 non-null      float64
 12  response_status     5 non-null      str    
dtypes: float64(4), int64(4), str(5)
memory usage: 8.1 KB


In [12]:
# Average monthly cost
avg_cost = df_v1['total_cost'].mean()

print(f"Average cost per question: ${avg_cost:.4f}")

projected_monthly_cost = avg_cost * 10000

print(f"Project monthly average cost: ${projected_monthly_cost:.2f}", )

Average cost per question: $0.0002
Project monthly average cost: $1.58


In [13]:
# Promt V2
PROMT_V2 = """
You are an academic assistant.
Answer the following student question in a clear, concise and accurate manner.
Use simple language suitable for undergraduate students.
Question: {Question}
"""

In [17]:
df_v2 = evaluate_prompt(
    questions,
    PROMT_V2,
    "v2",
    env['model_name'],
    env['input_token_price'],
    env['output_token_price']
)

print("Statistics of V2 evaluation")
print(df_v2['latency_seconds'].describe())

latency_stats_v2 = df_v2['latency_seconds'].agg(
    ['mean', 'median', "min", 'max', 'std']
).to_frame("value")

print("Selected parameters")
print(latency_stats_v2)

for _, row in df_v2.iterrows():
    print("PROMPT:")
    print(row["prompt"])

    print("\nRESPONSE:")
    print(row["response"])

    print("\n" + "="*80 + "\n")
# Average monthly cost
avg_cost_v2 = df_v2['total_cost'].mean()

print(f"Average cost per question: ${avg_cost_v2:.4f}")

projected_monthly_cost_v2 = avg_cost_v2 * 10000

print(f"Project monthly average cost: ${projected_monthly_cost_v2:.2f}", )

Statistics of V2 evaluation
count    5.000000
mean     3.502500
std      0.875562
min      2.311061
25%      3.063614
50%      3.528443
75%      4.016083
max      4.593302
Name: latency_seconds, dtype: float64
Selected parameters
           value
mean    3.502500
median  3.528443
min     2.311061
max     4.593302
std     0.875562
PROMPT:

You are an academic assistant.
Answer the following student question in a clear, concise and accurate manner.
Use simple language suitable for undergraduate students.
Question: what is overfitting in machine learning?


RESPONSE:
Overfitting in machine learning occurs when a model learns the training data too well, including its noise and outliers. This means the model becomes very complex and captures details that don’t generalize to new, unseen data. As a result, it performs very well on the training data but poorly on test or validation data because it hasn't learned the underlying patterns, just the specific examples. To prevent overfitting, techn

In [ ]:
PROMPT_V3 = """
You are an academic assistant.
Answer the student question and return ONLY valid JSON.
Use this exact JSON schema:
{{
    "answer": "string",
    "confidence": "high|medium|low",
    "key_points": ["string"]
}}

Question" {Question}
"""

In [21]:
df_v3 = evaluate_prompt(
    questions,
    PROMPT_V3,
    "v3",
    env['model_name'],
    env['input_token_price'],
    env['output_token_price']
)

print("Statistics of V3 evaluation")
print(df_v3['latency_seconds'].describe())

latency_stats_v3 = df_v3['latency_seconds'].agg(
    ['mean', 'median', "min", 'max', 'std']
).to_frame("value")

print("Selected parameters")
print(latency_stats_v3)

for _, row in df_v3.iterrows():
    print("PROMPT:")
    print(row["prompt"])

    print("\nRESPONSE:")
    print(row["response"])

    print("\n" + "="*80 + "\n")
# Average monthly cost
avg_cost_v3 = df_v3['total_cost'].mean()

print(f"Average cost per question: ${avg_cost_v3:.4f}")

projected_monthly_cost_v3 = avg_cost_v3 * 10000

print(f"Project monthly average cost: ${projected_monthly_cost_v3:.2f}", )

Statistics of V3 evaluation
count    5.000000
mean     2.557501
std      0.362847
min      2.243237
25%      2.254511
50%      2.390315
75%      2.934782
max      2.964658
Name: latency_seconds, dtype: float64
Selected parameters
           value
mean    2.557501
median  2.390315
min     2.243237
max     2.964658
std     0.362847
PROMPT:

You are an academic assistant.
Answer the studen question and return ONLY valid JSON.
Use this exact JSON schema:
{
    "answer": "string",
    "confidence": "high|medium|low",
    "key_points": ["string"]
}

Question" what is overfitting in machine learning?


RESPONSE:
{
    "answer": "Overfitting in machine learning occurs when a model learns the training data too well, capturing noise and outliers, which negatively affects its performance on new, unseen data. It reflects a model that is too complex, with high variance and low bias.",
    "confidence": "high",
    "key_points": [
        "Overfitting occurs when a model learns the training data exc

In [22]:
# JSON validation
import json

def is_valid_json(text):
    try:
        json.loads(text)
        return 1
    except:
        return 0


In [24]:
df_v3["valid_json"] = (
    df_v3["response"].apply(is_valid_json)
)

valid_json_rate = df_v3["valid_json"].mean()
print(valid_json_rate)

1.0


In [ ]:
# Prompt performance comparison

comparison_df = pd.DataFrame(
    [
        {
            "Prompt Version": "v1",
            "Avg Latency": df_v1["latency_seconds"].mean(),
            "Avg Input Tokens": df_v1["input_tokens"].mean(),
            "Avg Ouput Tokens": df_v1["output_tokens"].mean(),
            "Avg Total Tokens": df_v1["total_tokens"].mean(),
            "Avg Cost": df_v1["total_cost"].mean(),
            "JSON Valid": 0
        },
        {
            "Prompt Version": "v2",
            "Avg Latency": df_v2["latency_seconds"].mean(),
            "Avg Input Tokens": df_v2["input_tokens"].mean(),
            "Avg Ouput Tokens": df_v2["output_tokens"].mean(),
            "Avg Total Tokens": df_v2["total_tokens"].mean(),
            "Avg Cost": df_v2["total_cost"].mean(),
            "JSON Valid": 0
        },
        {
            "Prompt Version": "v3",
            "Avg Latency": df_v3["latency_seconds"].mean(),
            "Avg Input Tokens": df_v3["input_tokens"].mean(),
            "Avg Ouput Tokens": df_v3["output_tokens"].mean(),
            "Avg Total Tokens": df_v3["total_tokens"].mean(),
            "Avg Cost": df_v3["total_cost"].mean(),
            "JSON Valid": df_v3["valid_json"].mean()
        }
    ]
)

comparison_df

,Prompt Version,Avg Latency,Avg Input Tokens,Avg Ouput Tokens,Avg Cost,JSON Valid
0,v1,4.537414,24.6,270.0,0.000158,0.0
1,v2,3.502500,45.6,214.2,0.000129,0.0
2,v3,2.557501,68.6,132.6,0.000086,1.0


### Comparison of performance for three types of prompting

The comparison of performance for three types of prompting can be summarized as follows:
* **V1**: This version uses a basic prompt structure that provides minimal context. It is effective for straightforward tasks but may struggle with complex queries.
* **V2**: This version incorporates additional context. This instructs Language Model to assume an Academic Assistant's role.
It tends to perform better on tasks that require understanding nuanced instructions or generating more detailed responses.
* **V3**: This version follows **V2** but enhances further in terms of response format. The prompt expects the result in a JSON format, which is particularly useful for structured data extraction and integration with other systems. It is the most effective for tasks that require precise output formatting and data handling.

### Token consumption
* **V1**: Consumes the least number of tokens due to its simplicity, making it cost-effective for basic tasks.
* **V2**: Consumes more tokens than V1 because of the additional context provided, which can lead to higher costs but improved performance for complex tasks.
* **V3**: Consumes the most tokens due to the detailed instructions and expected output format. While it may be more expensive, it is justified for tasks that require structured data output and integration.